# GAFIME — Full API Tour 🚀

**GPU-Accelerated Feature Interaction Mining Engine**

This notebook is the canonical end-to-end reference for every public API
surface in `gafime`. Each section is self-contained — run them top to
bottom, or jump directly to the one that matches your usecase.

## Table of contents
1. Installation & backend check
2. 60-second quickstart — planted interaction signal
3. `EngineConfig` & `ComputeBudget` — tuning the search
4. Interpreting `DiagnosticReport` (interactions, stability, permutations, decision)
5. Real dataset — California housing regression
6. Classification — binary target with engineered interactions
7. `GafimeSelector` — scikit-learn pipeline drop-in
8. `GafimeStreamer` — VRAM-aware streaming over CSV/Parquet
9. CLI cheatsheet — `gafime --check`, `gafime --init`
10. Production tips

> **Hardware-aware.** GAFIME auto-selects the fastest available backend
> (CUDA → Metal → OpenMP/C++ core → NumPy). A CUDA-capable GPU will
> accelerate Pearson pair scoring up to **40×** over CPU at 200K×100.


## 1. Installation & backend check


Install the latest release from PyPI:

```bash
pip install gafime                # core + CUDA wheel (linux x86_64, macOS arm64, windows)
pip install 'gafime[sklearn]'     # + scikit-learn for GafimeSelector
pip install 'gafime[streaming]'   # + polars for GafimeStreamer
```

Then ask GAFIME which backend it would pick on this machine:


In [ ]:
import gafime
print('gafime version:', gafime.__version__)

# Equivalent to `gafime --check` on the command line
import numpy as np
from gafime.config import EngineConfig
from gafime.backends import resolve_backend

cfg = EngineConfig(backend='auto')
backend, warnings = resolve_backend(cfg, np.zeros((10, 2)), np.zeros(10))
info = backend.info()
print(f'Selected backend: {info.name}')
print(f'Device         : {info.device}')
print(f'Is GPU         : {info.is_gpu}')
for w in warnings:
    print('warn:', w)


## 2. 60-second quickstart

Plant an `X[:,0] * X[:,1]` interaction in a synthetic target and see if
GAFIME recovers it. This is the 'hello world' for feature interaction
mining.


In [ ]:
import numpy as np
from gafime import GafimeEngine, EngineConfig

rng = np.random.default_rng(42)
n_samples, n_features = 20_000, 20
X = rng.standard_normal((n_samples, n_features))
y = X[:, 3] * X[:, 7] + 0.3 * rng.standard_normal(n_samples)
feature_names = [f'f{i}' for i in range(n_features)]

# Pearson-only is the fast path on the CUDA backend
engine = GafimeEngine(EngineConfig(metric_names=('pearson',)))
report = engine.analyze(X, y, feature_names=feature_names)

print('backend :', report.backend.name, '/', report.backend.device)
print('decision:', report.decision.message)

# Top 5 strongest interactions
top = sorted(report.interactions,
             key=lambda r: max(abs(v) for v in r.metrics.values()),
             reverse=True)[:5]
for r in top:
    combo = ' × '.join(r.feature_names)
    score = max(abs(v) for v in r.metrics.values())
    print(f'  {combo:<20}  |pearson|={score:.3f}')


You should see `f3 × f7` at the top — that's the planted signal.


## 3. `EngineConfig` & `ComputeBudget`

Every knob that shapes the search is in these two frozen dataclasses.
Defaults are tuned for a 'small laptop' workload (≤50K × ≤50).

### `EngineConfig`
| field | default | meaning |
|---|---|---|
| `budget` | `ComputeBudget()` | combinatorial search limits (below) |
| `metric_names` | `('pearson','spearman','mutual_info','r2')` | metrics to score each combo with |
| `num_repeats` | `3` | bootstrap repeats for stability analysis |
| `permutation_tests` | `25` | y-permutations for null distribution |
| `random_seed` | `7` | determinism |
| `stability_std_threshold` | `0.10` | σ above which a metric is called unstable |
| `permutation_p_threshold` | `0.05` | significance cutoff |
| `mi_bins` | `16` | histogram bins for mutual-information |
| `backend` | `'auto'` | `'auto' | 'cuda' | 'metal' | 'cpu' | 'numpy'` |
| `device_id` | `0` | CUDA device index |

### `ComputeBudget`
| field | default | meaning |
|---|---|---|
| `max_comb_size` | `2` | up to k-way interactions (2 = pairs) |
| `max_combinations_per_k` | `5000` | cap per arity |
| `top_features_for_higher_k` | `50` | restrict k≥3 to top-N unary features |
| `max_generated_features` | `0` | extra derived columns to attempt |
| `keep_in_vram` | `True` | prefer GPU when available |
| `vram_budget_mb` | `6144` | VRAM ceiling (RTX 4060 8 GB leaves headroom) |

#### Recipes


In [ ]:
from gafime import EngineConfig, ComputeBudget

# A) Fast triage on a wide table: Pearson only, pairs only
fast = EngineConfig(
    metric_names=('pearson',),
    budget=ComputeBudget(max_comb_size=2, max_combinations_per_k=20_000),
    permutation_tests=10,
)

# B) Deep search: 3-way interactions, strict significance
deep = EngineConfig(
    metric_names=('pearson', 'spearman', 'mutual_info'),
    budget=ComputeBudget(max_comb_size=3, top_features_for_higher_k=30),
    permutation_tests=100,
    permutation_p_threshold=0.01,
)

# C) Force CPU for determinism/debugging
cpu_only = EngineConfig(backend='cpu', metric_names=('pearson', 'r2'))

for name, cfg in [('fast', fast), ('deep', deep), ('cpu_only', cpu_only)]:
    print(name, '->', cfg)


## 4. Interpreting `DiagnosticReport`

`engine.analyze()` returns a single `DiagnosticReport` with:

| attribute | type | what it is |
|---|---|---|
| `interactions` | `List[InteractionResult]` | combo + per-metric score |
| `stability` | `List[StabilityResult]` | mean/std of each metric over bootstrap repeats |
| `permutations` | `List[PermutationResult]` | empirical p-value per metric vs. y-permuted null |
| `decision` | `Decision` | `signal_detected` + human-readable message |
| `warnings` | `List[str]` | budget / memory warnings raised during the run |
| `backend` | `BackendInfo` | which backend actually ran |
| `feature_names` | `List[str]` | normalised feature names (auto-generated if not provided) |
| `config` | `EngineConfig` | echoed config for reproducibility |
| `.to_dict()` | method | JSON-serialisable snapshot |


In [ ]:
# Build a richer report and walk through every piece
cfg = EngineConfig(metric_names=('pearson', 'spearman'),
                   budget=ComputeBudget(max_comb_size=2),
                   permutation_tests=20)
report = GafimeEngine(cfg).analyze(X, y, feature_names=feature_names)

print('=== warnings ===')
for w in report.warnings:
    print(' -', w)

print()
print('=== decision ===')
print(' signal_detected:', report.decision.signal_detected)
print(' message        :', report.decision.message)

print()
print('=== backend ===')
print(vars(report.backend))


In [ ]:
# Merge interactions + stability + permutations into one pandas DataFrame
import pandas as pd

stab = {r.combo: r for r in report.stability}
perm = {r.combo: r for r in report.permutations}

rows = []
for r in report.interactions:
    row = {'combo': ' × '.join(r.feature_names), 'size': len(r.combo)}
    for m, v in r.metrics.items():
        row[f'{m}'] = v
        row[f'{m}_std'] = stab.get(r.combo, None) and stab[r.combo].metrics_std.get(m)
        row[f'{m}_p'] = perm.get(r.combo, None) and perm[r.combo].p_values.get(m)
    rows.append(row)

df = pd.DataFrame(rows)
# keep only significant + stable, sort by |pearson|
df['|pearson|'] = df['pearson'].abs()
keep = (df.get('pearson_p', 1.0) <= 0.05) & (df.get('pearson_std', 0.0) <= 0.10)
df.loc[keep].sort_values('|pearson|', ascending=False).head(10)


## 5. Real dataset — California housing regression

A classic tabular regression benchmark. We'll look for non-linear pair
interactions between the 8 numeric features that predict median house
value.


In [ ]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
X_df = data.data
y_arr = data.target.values
print(X_df.head())
print('shape:', X_df.shape)


In [ ]:
cfg = EngineConfig(
    metric_names=('pearson', 'spearman'),
    budget=ComputeBudget(max_comb_size=2),
    permutation_tests=30,
)
report = GafimeEngine(cfg).analyze(
    X_df.values, y_arr, feature_names=list(X_df.columns)
)

pairs = [(r, max(abs(v) for v in r.metrics.values()))
         for r in report.interactions if len(r.combo) == 2]
pairs.sort(key=lambda t: t[1], reverse=True)
print('Top 5 pair interactions:')
for r, s in pairs[:5]:
    name = ' × '.join(r.feature_names)
    pear = r.metrics.get('pearson', float('nan'))
    spear = r.metrics.get('spearman', float('nan'))
    print(f'  {name:<25}  pearson={pear:+.3f}  spearman={spear:+.3f}')


## 6. Classification with engineered interactions

Interactions surfaced by GAFIME plug straight into downstream models.
We'll binarise the California target, mine pairs, craft an interaction
matrix, and fit a logistic regression to compare against the raw
features.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

y_bin = (y_arr > np.median(y_arr)).astype(int)
Xtr, Xte, ytr, yte = train_test_split(
    X_df.values, y_bin, test_size=0.25, random_state=0, stratify=y_bin
)

# Baseline: raw features
scaler = StandardScaler().fit(Xtr)
base = LogisticRegression(max_iter=500).fit(scaler.transform(Xtr), ytr)
auc_base = roc_auc_score(yte, base.predict_proba(scaler.transform(Xte))[:, 1])

# Engineered: append top-5 pair products discovered by GAFIME
report = GafimeEngine(EngineConfig(metric_names=('pearson',),
                                   budget=ComputeBudget(max_comb_size=2))
                     ).analyze(Xtr, ytr, feature_names=list(X_df.columns))
top_pairs = [r.combo for r in sorted(
    (r for r in report.interactions if len(r.combo) == 2),
    key=lambda r: abs(r.metrics.get('pearson', 0.0)),
    reverse=True,
)[:5]]

def augment(A):
    extra = np.column_stack([A[:, i] * A[:, j] for i, j in top_pairs])
    return np.hstack([A, extra])

scaler2 = StandardScaler().fit(augment(Xtr))
lift = LogisticRegression(max_iter=500).fit(scaler2.transform(augment(Xtr)), ytr)
auc_lift = roc_auc_score(yte, lift.predict_proba(scaler2.transform(augment(Xte)))[:, 1])

print(f'AUC baseline          : {auc_base:.4f}')
print(f'AUC + GAFIME pairs    : {auc_lift:.4f}')
print(f'Δ                     : {auc_lift - auc_base:+.4f}')


## 7. `GafimeSelector` — sklearn pipeline drop-in

`GafimeSelector` is a `BaseEstimator + TransformerMixin` that evaluates
all pairwise interactions during `fit`, keeps the top-`k`, and appends
them as new columns during `transform`. It composes with
`Pipeline`, `GridSearchCV`, `cross_val_score`, etc.

**Constructor**
```python
GafimeSelector(
    k=10,                 # how many interactions to retain
    backend='auto',       # passthrough to the engine
    metric='pearson',     # 'pearson' | 'spearman' | 'r2'
    operator='multiply',  # how to combine the pair -> 'multiply' | 'add' | 'subtract' | 'divide'
    n_jobs=-1,
    verbose=False,
)
```
Fitted attributes: `top_interactions_` (list of `(i, j)`), `n_features_in_`.


In [ ]:
from gafime.sklearn import GafimeSelector
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

pipe = Pipeline([
    ('gafime', GafimeSelector(k=8, metric='pearson', operator='multiply')),
    ('clf', GradientBoostingClassifier(random_state=0)),
])
scores = cross_val_score(pipe, X_df.values, y_bin, cv=3, scoring='roc_auc')
print('CV ROC-AUC:', scores, '| mean:', scores.mean().round(4))

# Inspect what was selected on the full training set
pipe.fit(X_df.values, y_bin)
names = list(X_df.columns)
print('\nTop interactions kept by the selector:')
for i, j in pipe.named_steps['gafime'].top_interactions_:
    print(f'  {names[i]} × {names[j]}')


## 8. `GafimeStreamer` — VRAM-aware streaming

When your dataset exceeds RAM, stream chunks directly from CSV/Parquet
in sizes that fit your VRAM budget. Polars lazy frames do the disk I/O;
GAFIME sanitises each chunk to contiguous `float32` for `cudaMemcpy`.

```python
GafimeStreamer(file_path, target_cols=None, y_col=None)
    .total_rows                                   # cached row count
    .estimate_optimal_batch_size(vram_budget_gb)  # rows that fit your GPU
    .stream(batch_size=None)                      # yields X chunks
    .stream_with_target(batch_size=None)          # yields (X, y) chunks
```


In [ ]:
# Demo: write a synthetic parquet/csv and stream it
import tempfile, os
try:
    import polars as pl
    HAS_POLARS = True
except ImportError:
    HAS_POLARS = False

if not HAS_POLARS:
    print("polars not installed — `pip install 'gafime[streaming]'` to run this section")
else:
    tmp = tempfile.mkdtemp()
    path = os.path.join(tmp, 'demo.parquet')
    cols = {f'f{i}': rng.standard_normal(50_000).astype('float32') for i in range(8)}
    cols['y'] = (cols['f0'] * cols['f1'] + 0.5 * rng.standard_normal(50_000)).astype('float32')
    pl.DataFrame(cols).write_parquet(path)

    from gafime import GafimeStreamer
    streamer = GafimeStreamer(path, y_col='y')
    print('rows   :', streamer.total_rows)
    print('features:', streamer.n_features)
    print('optimal batch (6 GB VRAM):', streamer.estimate_optimal_batch_size(vram_budget_gb=6.0))

    for bi, (X_chunk, y_chunk) in enumerate(streamer.stream_with_target(batch_size=10_000)):
        print(f'  batch {bi}: X={X_chunk.shape} {X_chunk.dtype}, y={y_chunk.shape}')
        if bi == 2:
            break


## 9. CLI cheatsheet

After `pip install gafime`, the `gafime` executable is on `$PATH`:

```bash
gafime --version         # print installed version
gafime --check           # enumerate available backends and the one that would be picked
gafime --init            # write a starter notebook (`gafime_tutorial.ipynb`) to the cwd
gafime --init -o path.ipynb   # custom path
```

You can also invoke programmatically:


In [ ]:
from gafime import generate_tutorial
# generate_tutorial('my_starter.ipynb')   # uncomment to write file
print(generate_tutorial.__doc__)


## 10. Production tips

- **Pin a seed.** `EngineConfig(random_seed=…)` → deterministic bootstrap,
  permutations, and combo sampling across runs.
- **Keep it pair-only for speed.** `max_comb_size=2` + `metric_names=('pearson',)`
  activates the CUDA bucket fast path (≈40× CPU on ≥100K rows).
- **Gate on both p-value *and* stability.** A combo with low `permutations`
  p-value but large `stability.metrics_std` is a bootstrap artefact.
- **Reuse the engine.** `GafimeEngine` is stateless between calls but its
  CUDA backend caches device buffers keyed on `id(X)` — if you call
  `analyze` multiple times with the **same** `X` array, permutation passes
  skip upload + host-side centering.
- **Save the report.** `report.to_dict()` is JSON-serialisable after you
  convert the small handful of dataclasses with
  `dataclasses.asdict`, making it a drop-in artefact for MLflow /
  experiment tracking.
- **Pair with GPU-aware storage.** `GafimeStreamer` on Parquet → batched
  into `engine.analyze` lets you scan datasets larger than RAM without
  manual chunking.

### Reference summary (every public symbol)
| symbol | kind | lives in |
|---|---|---|
| `GafimeEngine(config=None).analyze(X, y, feature_names=None)` | class | `gafime.engine` |
| `EngineConfig(...)` | dataclass | `gafime.config` |
| `ComputeBudget(...)` | dataclass | `gafime.config` |
| `GafimeStreamer(file_path, target_cols=None, y_col=None)` | class | `gafime.io` |
| `GafimeSelector(k, backend, metric, operator, n_jobs, verbose)` | class | `gafime.sklearn` |
| `generate_tutorial(output_path)` | function | `gafime.tutorial` |
| `DiagnosticReport`, `InteractionResult`, `StabilityResult`, `PermutationResult`, `Decision` | dataclasses | `gafime.reporting` |
| `resolve_backend(config, X, y)` | function | `gafime.backends` |
| `__version__` | str | `gafime` |

Happy mining! 🧠⚡
